In [21]:
import pandas as pd
import numpy as np

In [22]:
df = pd.read_csv('../raw_data/train.csv')
df.head()

,Unnamed: 0,Name,Location,Year,Kilometers_Driven,Fuel_Type,Transmission,Owner_Type,Mileage,Engine,Power,Seats,New_Price,Price
0,1,Hyundai Creta 1.6 CRDi SX Option,Pune,2015,41000,Diesel,Manual,First,19.67 kmpl,1582 CC,126.2 bhp,5.0,NaN,12.50
1,2,Honda Jazz V,Chennai,2011,46000,Petrol,Manual,First,13 km/kg,1199 CC,88.7 bhp,5.0,8.61 Lakh,4.50
2,3,Maruti Ertiga VDI,Chennai,2012,87000,Diesel,Manual,First,20.77 kmpl,1248 CC,88.76 bhp,7.0,NaN,6.00
3,4,Audi A4 New 2.0 TDI Multitronic,Coimbatore,2013,40670,Diesel,Automatic,Second,15.2 kmpl,1968 CC,140.8 bhp,5.0,NaN,17.74
4,6,Nissan Micra Diesel XV,Jaipur,2013,86999,Diesel,Manual,First,23.08 kmpl,1461 CC,63.1 bhp,5.0,NaN,3.50


# B) Convert to Numerical values.
Doing this first so I can get the median for missing values.

In [23]:
df.drop('Unnamed: 0', axis=1, inplace=True)

In [24]:
df['Fuel_Type'].value_counts()

Fuel_Type
Diesel      3161
Petrol      2684
Electric       2
Name: count, dtype: int64

In [25]:
#convert to numeric
gasoline_density = 0.74
diesel_density = 0.83
def re(x):
    if "km/kg" in str(x['Mileage']):
        den = 1
        if x['Fuel_Type'] == 'Diesel':
            den = diesel_density
        if x['Fuel_Type'] == 'Petrol':
            den = gasoline_density
        return float(x['Mileage'].replace("km/kg", "").strip()) * den
    if "kmpl" in str(x['Mileage']):
        return float(x['Mileage'].replace("kmpl", "").strip()) 

df['Mileage'] = df.apply(re, axis=1)

df.head()

,Name,Location,Year,Kilometers_Driven,Fuel_Type,Transmission,Owner_Type,Mileage,Engine,Power,Seats,New_Price,Price
0,Hyundai Creta 1.6 CRDi SX Option,Pune,2015,41000,Diesel,Manual,First,19.67,1582 CC,126.2 bhp,5.0,NaN,12.50
1,Honda Jazz V,Chennai,2011,46000,Petrol,Manual,First,9.62,1199 CC,88.7 bhp,5.0,8.61 Lakh,4.50
2,Maruti Ertiga VDI,Chennai,2012,87000,Diesel,Manual,First,20.77,1248 CC,88.76 bhp,7.0,NaN,6.00
3,Audi A4 New 2.0 TDI Multitronic,Coimbatore,2013,40670,Diesel,Automatic,Second,15.20,1968 CC,140.8 bhp,5.0,NaN,17.74
4,Nissan Micra Diesel XV,Jaipur,2013,86999,Diesel,Manual,First,23.08,1461 CC,63.1 bhp,5.0,NaN,3.50


In [26]:
df['Engine'] = df['Engine'].str.replace("CC", "").str.strip().astype(float)
df['Power'] = df['Power'].str.replace("bhp", "").str.strip().astype(float)
df.dtypes

Name                  object
Location              object
Year                   int64
Kilometers_Driven      int64
Fuel_Type             object
Transmission          object
Owner_Type            object
Mileage              float64
Engine               float64
Power                float64
Seats                float64
New_Price             object
Price                float64
dtype: object

# A) Handle Missing Values

In [27]:
df.describe()

,Year,Kilometers_Driven,Mileage,Engine,Power,Seats,Price
count,5847.000000,5.847000e+03,5845.000000,5811.000000,5811.000000,5809.000000,5847.000000
mean,2013.448435,5.841013e+04,18.157163,1631.552573,113.803144,5.286452,9.653742
std,3.194949,9.237971e+04,4.360093,601.972587,53.896719,0.806668,11.275966
min,1998.000000,1.710000e+02,0.000000,72.000000,34.200000,2.000000,0.440000
25%,2012.000000,3.346750e+04,15.200000,1198.000000,78.000000,5.000000,3.550000
50%,2014.000000,5.257600e+04,18.190000,1497.000000,98.600000,5.000000,5.750000
75%,2016.000000,7.249050e+04,21.100000,1991.000000,139.010000,5.000000,10.250000
max,2019.000000,6.500000e+06,28.400000,5998.000000,560.000000,10.000000,160.000000


In [28]:
df.isna().sum()

Name                    0
Location                0
Year                    0
Kilometers_Driven       0
Fuel_Type               0
Transmission            0
Owner_Type              0
Mileage                 2
Engine                 36
Power                  36
Seats                  38
New_Price            5032
Price                   0
dtype: int64

In [29]:
df.shape

(5847, 13)

In [30]:
# The majority of new price is missing, so there isn't any meaningful way to impute the value.
df.drop(labels=['New_Price'], axis=1, inplace=True)

In [31]:
# using median due to outliers
cols = ['Mileage', 'Engine', 'Power', 'Seats']
for col in cols:
    df[col] = df[col].fillna(df[col].median())

df.isna().sum()


Name                 0
Location             0
Year                 0
Kilometers_Driven    0
Fuel_Type            0
Transmission         0
Owner_Type           0
Mileage              0
Engine               0
Power                0
Seats                0
Price                0
dtype: int64

In [32]:
# There's still useful features here without new price
df.select_dtypes(include='number').corr()

,Year,Kilometers_Driven,Mileage,Engine,Power,Seats,Price
Year,1.000000,-0.169514,0.300648,-0.065935,0.017388,0.010770,0.299947
Kilometers_Driven,-0.169514,1.000000,-0.061988,0.092984,0.033142,0.082992,-0.008592
Mileage,0.300648,-0.061988,1.000000,-0.621756,-0.522679,-0.319866,-0.331289
Engine,-0.065935,0.092984,-0.621756,1.000000,0.864941,0.400310,0.655371
Power,0.017388,0.033142,-0.522679,0.864941,1.000000,0.098841,0.770924
Seats,0.010770,0.082992,-0.319866,0.400310,0.098841,1.000000,0.053739
Price,0.299947,-0.008592,-0.331289,0.655371,0.770924,0.053739,1.000000


# C) Hot Encoding

In [33]:
cols = ['Fuel_Type', 'Transmission', 'Owner_Type']
for col in cols:
    encoding = pd.get_dummies(df[col], drop_first=True, dtype=int)
    df = df.drop(labels=[col], axis=1).join(encoding)

df.head()

,Name,Location,Year,Kilometers_Driven,Mileage,Engine,Power,Seats,Price,Electric,Petrol,Manual,Fourth & Above,Second,Third
0,Hyundai Creta 1.6 CRDi SX Option,Pune,2015,41000,19.67,1582.0,126.20,5.0,12.50,0,0,1,0,0,0
1,Honda Jazz V,Chennai,2011,46000,9.62,1199.0,88.70,5.0,4.50,0,1,1,0,0,0
2,Maruti Ertiga VDI,Chennai,2012,87000,20.77,1248.0,88.76,7.0,6.00,0,0,1,0,0,0
3,Audi A4 New 2.0 TDI Multitronic,Coimbatore,2013,40670,15.20,1968.0,140.80,5.0,17.74,0,0,0,0,1,0
4,Nissan Micra Diesel XV,Jaipur,2013,86999,23.08,1461.0,63.10,5.0,3.50,0,0,1,0,0,0


# D) Enrichment

In [151]:
from datetime import datetime

year = datetime.now().year

df['Age'] = year - df['Year']

df.head()

,Name,Location,Year,Kilometers_Driven,Mileage,Engine,Power,Seats,Price,Electric,Petrol,Manual,Fourth & Above,Second,Third,Age
0,Hyundai Creta 1.6 CRDi SX Option,Pune,2015,41000,19.67,1582.0,126.20,5.0,12.50,0,0,1,0,0,0,10
1,Honda Jazz V,Chennai,2011,46000,9.62,1199.0,88.70,5.0,4.50,0,1,1,0,0,0,14
2,Maruti Ertiga VDI,Chennai,2012,87000,20.77,1248.0,88.76,7.0,6.00,0,0,1,0,0,0,13
3,Audi A4 New 2.0 TDI Multitronic,Coimbatore,2013,40670,15.20,1968.0,140.80,5.0,17.74,0,0,0,0,1,0,12
4,Nissan Micra Diesel XV,Jaipur,2013,86999,23.08,1461.0,63.10,5.0,3.50,0,0,1,0,0,0,12


In [152]:
df['Kilometers_Per_Year'] = df['Kilometers_Driven'] / df['Age']

df.head()

,Name,Location,Year,Kilometers_Driven,Mileage,Engine,Power,Seats,Price,Electric,Petrol,Manual,Fourth & Above,Second,Third,Age,Kilometers_Per_Year
0,Hyundai Creta 1.6 CRDi SX Option,Pune,2015,41000,19.67,1582.0,126.20,5.0,12.50,0,0,1,0,0,0,10,4100.000000
1,Honda Jazz V,Chennai,2011,46000,9.62,1199.0,88.70,5.0,4.50,0,1,1,0,0,0,14,3285.714286
2,Maruti Ertiga VDI,Chennai,2012,87000,20.77,1248.0,88.76,7.0,6.00,0,0,1,0,0,0,13,6692.307692
3,Audi A4 New 2.0 TDI Multitronic,Coimbatore,2013,40670,15.20,1968.0,140.80,5.0,17.74,0,0,0,0,1,0,12,3389.166667
4,Nissan Micra Diesel XV,Jaipur,2013,86999,23.08,1461.0,63.10,5.0,3.50,0,0,1,0,0,0,12,7249.916667


In [34]:
df.to_csv("../clean_data/clean_car_data.csv", index=False)

# E) Querying

In [ ]:
q = df[df['Year'] > 2014]\
    [['Kilometers_Driven', 'Location', 'Mileage', 'Engine', 'Power', 'Seats', 'Price']]\
    .groupby('Location')\
    .median()\
    .reset_index()\
    .rename(columns={'Mileage': 'Mileage (kmpl)', 'Engine': 'Engine (CC)', 'Power': 'Power (bhp)'})\
    .sort_values('Price', ascending=False)

q

,Location,Kilometers_Driven,Mileage (kmpl),Engine (CC),Power (bhp),Seats,Price
1,Bangalore,38000.0,18.100,1582.0,122.00,5.0,11.500
3,Coimbatore,36931.0,18.500,1498.0,105.00,5.0,9.600
5,Hyderabad,45000.0,20.140,1477.0,88.80,5.0,7.950
9,Mumbai,26000.0,18.600,1497.0,103.50,5.0,7.750
0,Ahmedabad,42075.0,19.995,1451.0,94.85,5.0,7.750
7,Kochi,37250.0,18.900,1461.0,91.15,5.0,7.540
4,Delhi,40000.0,18.900,1396.0,88.73,5.0,6.950
2,Chennai,39000.0,18.990,1396.0,88.70,5.0,6.900
10,Pune,41873.5,19.870,1248.0,88.50,5.0,6.625
6,Jaipur,44285.5,21.100,1248.0,82.00,5.0,5.850
